# 🖼️ Image Captioning con CNN + LSTM
**Procesamiento del Lenguaje Natural para IA**

Arquitectura basada en *Show and Tell* (Vinyals et al., 2015):
- **Encoder**: ResNet-50 preentrenada en ImageNet (pesos congelados)
- **Embeddings**: GloVe 100d preentrenados
- **Decoder**: LSTM que genera la descripción palabra a palabra
- **Dataset**: Flickr8k (8.000 imágenes, 5 captions cada una)

> ⚠️ Antes de ejecutar: **Entorno de ejecución → Cambiar tipo → T4 GPU**

## 0. Instalaciones y librerías

In [1]:
# Instalamos las dependencias necesarias
# nltk: para calcular la métrica BLEU
# kaggle: para descargar Flickr8k automáticamente
!pip install nltk kaggle --quiet

import os
import json
import random
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

import nltk
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Configuración del dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Usando dispositivo: cpu


## 1. Descarga del dataset Flickr8k

Usamos Kaggle para descargarlo. Necesitas subir tu `kaggle.json` (API key).
Puedes obtenerla en: **kaggle.com → tu perfil → Settings → API → Create New Token**

In [2]:
# Opción A: subir kaggle.json manualmente
from google.colab import files
print('Sube tu kaggle.json:')
files.upload()

# Colocamos la API key donde Kaggle la espera
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Descargamos Flickr8k (~1GB, tarda ~2-3 min en Colab)
!kaggle datasets download -d adityajn105/flickr8k
!unzip -q flickr8k.zip -d flickr8k

print('Dataset descargado y descomprimido ✓')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Comprobamos la estructura del dataset
!ls flickr8k/
!echo '---'
!ls flickr8k/Images/ | head -5
!echo '---'
!head -3 flickr8k/captions.txt

## 2. Preprocesado de captions y vocabulario

In [ ]:
# ============================================================
# PASO 1: Cargamos y limpiamos las captions del dataset
# Cada imagen tiene 5 descripciones asociadas
# ============================================================

def load_captions(captions_file):
    """
    Lee el fichero de captions y devuelve un diccionario
    {nombre_imagen: [caption1, caption2, ...]}
    """
    captions = {}
    with open(captions_file, 'r') as f:
        next(f)  # saltamos la cabecera 'image,caption'
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Separamos solo por la primera coma para no romper captions con comas
            img_name, caption = line.split(',', 1)
            img_name = img_name.strip()
            caption = caption.strip().lower()
            if img_name not in captions:
                captions[img_name] = []
            captions[img_name].append(caption)
    return captions


def clean_caption(caption):
    """
    Limpieza básica: minúsculas, eliminar puntuación y números,
    quedarnos solo con palabras de más de 1 carácter.
    """
    # Eliminamos caracteres que no sean letras o espacios
    caption = ''.join(c for c in caption if c.isalpha() or c == ' ')
    # Normalizamos espacios y filtramos tokens cortos
    tokens = [w for w in caption.split() if len(w) > 1]
    return ' '.join(tokens)


# Cargamos y limpiamos
raw_captions = load_captions('flickr8k/captions.txt')
clean_captions = {}
for img, caps in raw_captions.items():
    clean_captions[img] = [clean_caption(c) for c in caps]

# Mostramos un ejemplo
ejemplo_img = list(clean_captions.keys())[0]
print(f'Imagen: {ejemplo_img}')
for i, cap in enumerate(clean_captions[ejemplo_img]):
    print(f'  Caption {i+1}: {cap}')

print(f'\nTotal de imágenes: {len(clean_captions)}')

In [ ]:
# ============================================================
# PASO 2: Construimos el vocabulario
# Solo incluimos palabras que aparecen >= MIN_FREQ veces
# para evitar tokens raros que el modelo no puede aprender bien.
# Añadimos tokens especiales:
#   <PAD>: relleno para igualar longitudes en un batch
#   <SOS>: Start Of Sequence - indica al LSTM que empiece
#   <EOS>: End Of Sequence - indica que la frase ha terminado
#   <UNK>: Unknown - palabras fuera del vocabulario
# ============================================================

MIN_FREQ = 5  # umbral mínimo de frecuencia

class Vocabulary:
    def __init__(self, min_freq=5):
        self.min_freq = min_freq
        # Tokens especiales con índices fijos
        self.word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}

    def build(self, captions_dict):
        """Construye el vocabulario a partir de todas las captions."""
        # Contamos la frecuencia de cada palabra en todas las captions
        counter = Counter()
        for caps in captions_dict.values():
            for cap in caps:
                counter.update(cap.split())

        # Solo añadimos palabras que superen el umbral de frecuencia
        for word, freq in counter.items():
            if freq >= self.min_freq:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

        print(f'Vocabulario construido: {len(self.word2idx)} palabras'
              f' (min_freq={self.min_freq})')

    def encode(self, caption):
        """Convierte una frase en lista de índices, con <SOS> y <EOS>."""
        unk = self.word2idx['<UNK>']
        return (
            [self.word2idx['<SOS>']] +
            [self.word2idx.get(w, unk) for w in caption.split()] +
            [self.word2idx['<EOS>']]
        )

    def decode(self, indices):
        """Convierte una lista de índices en frase legible."""
        words = []
        for idx in indices:
            word = self.idx2word.get(idx, '<UNK>')
            if word in ('<SOS>', '<PAD>'):
                continue
            if word == '<EOS>':
                break
            words.append(word)
        return ' '.join(words)

    def __len__(self):
        return len(self.word2idx)


vocab = Vocabulary(min_freq=MIN_FREQ)
vocab.build(clean_captions)

# Comprobamos que la codificación funciona
test_cap = clean_captions[ejemplo_img][0]
encoded = vocab.encode(test_cap)
decoded = vocab.decode(encoded)
print(f'\nOriginal : {test_cap}')
print(f'Codificado: {encoded[:8]}...')
print(f'Decodificado: {decoded}')

In [ ]:
# ============================================================
# PASO 3: División train / val / test
# Usamos los splits estándar de Flickr8k:
# 6000 train, 1000 val, 1000 test
# ============================================================

all_images = sorted(list(clean_captions.keys()))
random.seed(42)  # fijamos la semilla para reproducibilidad
random.shuffle(all_images)

train_imgs = all_images[:6000]
val_imgs   = all_images[6000:7000]
test_imgs  = all_images[7000:]

print(f'Train: {len(train_imgs)} imágenes')
print(f'Val:   {len(val_imgs)} imágenes')
print(f'Test:  {len(test_imgs)} imágenes')

## 3. Dataset y DataLoader

In [ ]:
# ============================================================
# Definimos el Dataset de PyTorch.
# En cada __getitem__ devolvemos:
#   - imagen: tensor [3, 224, 224] normalizado para ResNet
#   - caption: tensor de índices del vocabulario
# Durante el entrenamiento usamos una sola caption aleatoria
# de las 5 disponibles por imagen (data augmentation ligero).
# ============================================================

# Transformaciones de imagen para ResNet:
# - Redimensionamos a 256 y hacemos crop central de 224
# - Normalizamos con media y std de ImageNet
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),       # augmentation: crop aleatorio en train
    transforms.RandomHorizontalFlip(), # augmentation: flip horizontal
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),        # en val/test, crop fijo
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


class Flickr8kDataset(Dataset):
    def __init__(self, image_names, captions_dict, vocab,
                 img_dir, transform=None, mode='train'):
        self.image_names   = image_names
        self.captions_dict = captions_dict
        self.vocab         = vocab
        self.img_dir       = img_dir
        self.transform     = transform
        self.mode          = mode  # 'train' o 'val'/'test'

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)

        # Cargamos la imagen en RGB (algunas imágenes en Flickr son RGBA o L)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # En train usamos una caption aleatoria de las 5 disponibles
        # En val/test usamos siempre la primera (evaluación consistente)
        caps = self.captions_dict[img_name]
        if self.mode == 'train':
            caption = random.choice(caps)
        else:
            caption = caps[0]

        caption_tensor = torch.tensor(self.vocab.encode(caption),
                                      dtype=torch.long)
        return image, caption_tensor


def collate_fn(batch):
    """
    Función de collate personalizada para el DataLoader.
    Las captions tienen longitudes distintas, así que las
    rellenamos con <PAD> (índice 0) hasta la longitud máxima del batch.
    """
    images, captions = zip(*batch)
    images = torch.stack(images, dim=0)

    # Longitud máxima de caption en este batch
    max_len = max(cap.size(0) for cap in captions)

    # Rellenamos con ceros (<PAD>) las captions más cortas
    padded = torch.zeros(len(captions), max_len, dtype=torch.long)
    for i, cap in enumerate(captions):
        padded[i, :cap.size(0)] = cap

    return images, padded


# Creamos los datasets y dataloaders
IMG_DIR = 'flickr8k/Images'
BATCH_SIZE = 64

train_dataset = Flickr8kDataset(train_imgs, clean_captions, vocab,
                                IMG_DIR, train_transform, mode='train')
val_dataset   = Flickr8kDataset(val_imgs,  clean_captions, vocab,
                                IMG_DIR, val_transform,   mode='val')
test_dataset  = Flickr8kDataset(test_imgs, clean_captions, vocab,
                                IMG_DIR, val_transform,   mode='test')

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  collate_fn=collate_fn,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, collate_fn=collate_fn,
                          num_workers=2, pin_memory=True)

print(f'Batches en train: {len(train_loader)}')
print(f'Batches en val:   {len(val_loader)}')

# Comprobamos que un batch tiene la forma correcta
imgs_batch, caps_batch = next(iter(train_loader))
print(f'\nForma de un batch de imágenes: {imgs_batch.shape}')  # [64, 3, 224, 224]
print(f'Forma de un batch de captions: {caps_batch.shape}')    # [64, max_len]

## 4. Embeddings GloVe preentrenados

In [ ]:
# ============================================================
# Descargamos GloVe 100d y construimos la matriz de embeddings.
# Cada palabra del vocabulario se inicializa con su vector GloVe.
# Las palabras fuera de GloVe (tokens especiales o raras)
# se inicializan aleatoriamente.
# ============================================================

EMBED_DIM = 100  # dimensión de GloVe

# Descargamos el fichero GloVe (~330MB)
!wget -q --show-progress https://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip -d glove
print('GloVe descargado ✓')

In [ ]:
def load_glove(glove_path, vocab, embed_dim):
    """
    Carga los vectores GloVe y construye una matriz de embeddings
    de tamaño [vocab_size, embed_dim].
    Las palabras no encontradas en GloVe se inicializan con
    una distribución normal pequeña.
    """
    vocab_size = len(vocab)
    # Inicialización aleatoria para todas las palabras
    embedding_matrix = np.random.normal(0, 0.1, (vocab_size, embed_dim))
    # <PAD> siempre es vector cero
    embedding_matrix[0] = np.zeros(embed_dim)

    found = 0
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.split()
            word = parts[0]
            if word in vocab.word2idx:
                vector = np.array(parts[1:], dtype=np.float32)
                embedding_matrix[vocab.word2idx[word]] = vector
                found += 1

    print(f'Palabras encontradas en GloVe: {found}/{vocab_size} '
          f'({100*found/vocab_size:.1f}%)')
    return torch.FloatTensor(embedding_matrix)


glove_path = f'glove/glove.6B.{EMBED_DIM}d.txt'
embedding_matrix = load_glove(glove_path, vocab, EMBED_DIM)
print(f'Matriz de embeddings: {embedding_matrix.shape}')

## 5. Modelo: Encoder (ResNet-50) + Decoder (LSTM)

In [ ]:
# ============================================================
# ENCODER: ResNet-50 preentrenada
# Eliminamos la capa de clasificación final y añadimos
# una proyección lineal para adaptar las features de 2048
# dimensiones al espacio de embeddings (100d).
# Los pesos de ResNet se congelan para ahorrar tiempo y
# evitar overfitting con un dataset pequeño.
# ============================================================

class EncoderCNN(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        # Cargamos ResNet-50 preentrenada en ImageNet
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        # Congelamos todos los parámetros de ResNet:
        # no se actualizarán durante el backprop
        for param in resnet.parameters():
            param.requires_grad = False

        # Quitamos la última capa de clasificación (fc)
        # y nos quedamos con todo lo anterior
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)

        # Capa de proyección: 2048 -> embed_dim
        # Esta capa SÍ se entrena
        self.projection = nn.Linear(resnet.fc.in_features, embed_dim)
        self.bn = nn.BatchNorm1d(embed_dim)

    def forward(self, images):
        # images: [batch, 3, 224, 224]
        with torch.no_grad():
            features = self.resnet(images)   # [batch, 2048, 1, 1]
        features = features.squeeze(-1).squeeze(-1)  # [batch, 2048]
        features = self.bn(self.projection(features)) # [batch, embed_dim]
        return features


# ============================================================
# DECODER: LSTM
# Recibe el vector visual como estado inicial y genera
# la descripción palabra a palabra.
# En cada paso t:
#   input  = embedding de la palabra anterior
#   output = distribución de probabilidad sobre el vocabulario
# ============================================================

class DecoderLSTM(nn.Module):
    def __init__(self, embed_dim, hidden_dim, vocab_size,
                 embedding_matrix, dropout=0.5):
        super().__init__()

        # Capa de embeddings inicializada con GloVe
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight.data.copy_(embedding_matrix)
        # Permitimos fine-tuning de los embeddings durante el entrenamiento
        self.embedding.weight.requires_grad = True

        # LSTM: recibe embed_dim en cada paso
        self.lstm = nn.LSTM(embed_dim, hidden_dim,
                            batch_first=True, dropout=dropout if dropout > 0 else 0)

        # Capa de salida: proyectamos el hidden state al vocabulario
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, features, captions):
        """
        Modo entrenamiento con teacher forcing:
        en cada paso t damos la palabra real t (no la predicha).
        features:  [batch, embed_dim]  - vector visual del encoder
        captions:  [batch, seq_len]    - secuencia de índices (con <SOS>)
        """
        # No usamos el <EOS> como input (el modelo lo predice, no lo consume)
        embeddings = self.dropout(self.embedding(captions[:, :-1]))
        # [batch, seq_len-1, embed_dim]

        # El vector visual se usa como estado inicial de la LSTM
        # Reshape: [1, batch, hidden_dim]
        h0 = features.unsqueeze(0)
        # Proyectamos también para el cell state
        c0 = torch.zeros_like(h0)

        # Pasamos toda la secuencia de una vez (teacher forcing)
        output, _ = self.lstm(embeddings, (h0, c0))
        # output: [batch, seq_len-1, hidden_dim]

        # Proyectamos al vocabulario
        logits = self.fc(self.dropout(output))
        # logits: [batch, seq_len-1, vocab_size]
        return logits


# ============================================================
# Modelo completo: Encoder + Decoder
# ============================================================

class ImageCaptioningModel(nn.Module):
    def __init__(self, embed_dim, hidden_dim, vocab_size, embedding_matrix):
        super().__init__()
        self.encoder = EncoderCNN(embed_dim)
        self.decoder = DecoderLSTM(embed_dim, hidden_dim,
                                   vocab_size, embedding_matrix)

    def forward(self, images, captions):
        features = self.encoder(images)
        logits   = self.decoder(features, captions)
        return logits


# Hiperparámetros del modelo
HIDDEN_DIM = 512

model = ImageCaptioningModel(
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    vocab_size=len(vocab),
    embedding_matrix=embedding_matrix
).to(device)

# Resumen del modelo: número de parámetros entrenables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parámetros entrenables: {trainable:,} / {total:,} total')

## 6. Entrenamiento

In [ ]:
# ============================================================
# Configuración del entrenamiento:
# - CrossEntropyLoss ignorando el padding (<PAD> = índice 0)
# - Adam como optimizador (solo sobre parámetros entrenables)
# - ReduceLROnPlateau: reduce el LR si la val_loss no mejora
# ============================================================

# Conectamos Google Drive para guardar checkpoints
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/image_captioning_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Loss: ignoramos el token de padding en el cálculo
criterion = nn.CrossEntropyLoss(ignore_index=vocab.word2idx['<PAD>'])

# Optimizador: solo actualizamos los parámetros entrenables
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=4e-4
)

# Scheduler: reduce el LR a la mitad si val_loss no mejora en 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2, verbose=True
)

In [ ]:
# ============================================================
# Funciones de entrenamiento y validación por epoch
# ============================================================

def train_epoch(model, loader, criterion, optimizer, device):
    """Ejecuta un epoch completo de entrenamiento."""
    model.train()
    total_loss = 0.0

    for images, captions in loader:
        images   = images.to(device)
        captions = captions.to(device)

        optimizer.zero_grad()

        # Forward pass
        logits = model(images, captions)
        # logits:   [batch, seq_len-1, vocab_size]
        # targets:  [batch, seq_len-1]  (quitamos el <SOS>)

        # Reorganizamos para CrossEntropyLoss
        logits  = logits.reshape(-1, logits.size(-1))  # [batch*(seq-1), vocab]
        targets = captions[:, 1:].reshape(-1)           # [batch*(seq-1)]

        loss = criterion(logits, targets)
        loss.backward()

        # Gradient clipping: evita exploding gradients en la LSTM
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def val_epoch(model, loader, criterion, device):
    """Evalúa el modelo en el conjunto de validación."""
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, captions in loader:
            images   = images.to(device)
            captions = captions.to(device)
            logits   = model(images, captions)
            logits   = logits.reshape(-1, logits.size(-1))
            targets  = captions[:, 1:].reshape(-1)
            total_loss += criterion(logits, targets).item()

    return total_loss / len(loader)

In [ ]:
# ============================================================
# Bucle principal de entrenamiento
# Guardamos el mejor modelo (menor val_loss) en Drive
# ============================================================

NUM_EPOCHS = 20
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss   = val_epoch(model, val_loader, criterion, device)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f}')

    # Guardamos el modelo si es el mejor hasta ahora
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint = {
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss': val_loss,
            'vocab': vocab
        }
        torch.save(checkpoint,
                   f'{CHECKPOINT_DIR}/best_model.pt')
        print(f'  → Nuevo mejor modelo guardado (val_loss={val_loss:.4f})')

print('\nEntrenamiento completado ✓')

In [ ]:
# Visualizamos las curvas de pérdida
plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'],   label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Curvas de entrenamiento')
plt.legend()
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/training_curves.png', dpi=150)
plt.show()

## 7. Generación de captions (Greedy y Beam Search)

In [ ]:
# ============================================================
# Cargamos el mejor modelo guardado en Drive
# ============================================================

checkpoint = torch.load(f'{CHECKPOINT_DIR}/best_model.pt',
                        map_location=device)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f'Modelo cargado (epoch {checkpoint["epoch"]}, '
      f'val_loss={checkpoint["val_loss"]:.4f})')

In [ ]:
# ============================================================
# GREEDY SEARCH
# En cada paso seleccionamos la palabra con mayor probabilidad.
# Simple y rápido, pero no siempre da la mejor secuencia global.
# ============================================================

@torch.no_grad()
def generate_greedy(model, image_tensor, vocab, max_len=30):
    """
    Genera una descripción palabra a palabra eligiendo
    siempre el token más probable.
    """
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)  # [1, 3, 224, 224]

    # Extraemos el vector visual
    features = model.encoder(image_tensor)  # [1, embed_dim]

    # Inicializamos con <SOS>
    input_token = torch.tensor([[vocab.word2idx['<SOS>']]], device=device)
    h = features.unsqueeze(0)  # [1, 1, embed_dim]
    c = torch.zeros_like(h)

    generated = []
    for _ in range(max_len):
        emb = model.decoder.embedding(input_token)  # [1, 1, embed_dim]
        out, (h, c) = model.decoder.lstm(emb, (h, c))
        logits = model.decoder.fc(out.squeeze(1))   # [1, vocab_size]
        next_token = logits.argmax(dim=-1)           # [1]

        word = vocab.idx2word[next_token.item()]
        if word == '<EOS>':
            break
        generated.append(word)
        input_token = next_token.unsqueeze(0)       # [1, 1]

    return ' '.join(generated)


# ============================================================
# BEAM SEARCH
# Mantiene las k mejores hipótesis en cada paso.
# Produce resultados más coherentes que greedy.
# ============================================================

@torch.no_grad()
def generate_beam(model, image_tensor, vocab, beam_size=5, max_len=30):
    """
    Genera una descripción usando beam search.
    Cada hipótesis es una tupla: (log_prob_acumulada, tokens, h, c)
    """
    model.eval()
    image_tensor = image_tensor.unsqueeze(0).to(device)
    features = model.encoder(image_tensor)  # [1, embed_dim]

    h0 = features.unsqueeze(0)  # [1, 1, hidden_dim]
    c0 = torch.zeros_like(h0)

    sos = vocab.word2idx['<SOS>']
    eos = vocab.word2idx['<EOS>']

    # Hipótesis iniciales: [(log_prob, tokens, h, c)]
    beams = [(0.0, [sos], h0, c0)]
    completed = []

    for _ in range(max_len):
        new_beams = []
        for log_prob, tokens, h, c in beams:
            last_token = torch.tensor([[tokens[-1]]], device=device)
            emb = model.decoder.embedding(last_token)  # [1, 1, embed_dim]
            out, (h_new, c_new) = model.decoder.lstm(emb, (h, c))
            logits = model.decoder.fc(out.squeeze(1))  # [1, vocab_size]
            log_probs = torch.log_softmax(logits, dim=-1).squeeze(0)  # [vocab]

            # Tomamos los top beam_size candidatos
            top_log_probs, top_tokens = log_probs.topk(beam_size)

            for lp, tok in zip(top_log_probs.tolist(), top_tokens.tolist()):
                new_seq = tokens + [tok]
                new_log_prob = log_prob + lp
                if tok == eos:
                    completed.append((new_log_prob, new_seq))
                else:
                    new_beams.append((new_log_prob, new_seq, h_new, c_new))

        # Ordenamos y mantenemos solo los beam_size mejores
        new_beams.sort(key=lambda x: x[0], reverse=True)
        beams = new_beams[:beam_size]

        if not beams:
            break

    # Si no hay ninguna secuencia completa, tomamos la mejor del beam
    if not completed:
        completed = [(b[0], b[1]) for b in beams]

    # Normalizamos la log-prob por longitud y elegimos la mejor
    best_log_prob, best_tokens = max(
        completed,
        key=lambda x: x[0] / max(len(x[1]), 1)
    )
    return vocab.decode(best_tokens)

## 8. Evaluación con BLEU

In [ ]:
# ============================================================
# BLEU (Bilingual Evaluation Understudy)
# Compara n-gramas entre la descripción generada y las 5
# referencias humanas. Es la métrica estándar en los papers.
# BLEU-1: unigramas  | BLEU-4: 1-2-3-4 gramas combinados
# ============================================================

def evaluate_bleu(model, image_names, captions_dict, vocab,
                  img_dir, transform, device, beam_size=5, n_samples=500):
    """
    Calcula BLEU-1 y BLEU-4 sobre un subconjunto de imágenes.
    Usamos n_samples imágenes para que sea manejable en tiempo.
    """
    model.eval()
    smoothing = SmoothingFunction().method1  # evita BLEU=0 por n-gramas no encontrados

    references_all = []  # lista de listas de referencias tokenizadas
    hypotheses_all = []  # lista de hipótesis tokenizadas

    sample_names = random.sample(image_names, min(n_samples, len(image_names)))

    for img_name in sample_names:
        # Cargamos y transformamos la imagen
        img_path = os.path.join(img_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        image_tensor = transform(image)

        # Generamos la descripción
        hypothesis = generate_beam(model, image_tensor, vocab,
                                   beam_size=beam_size)

        # Las 5 referencias humanas
        refs = [clean_caption(cap).split()
                for cap in captions_dict[img_name]]

        references_all.append(refs)
        hypotheses_all.append(hypothesis.split())

    # BLEU-1: solo unigramas
    bleu1 = corpus_bleu(references_all, hypotheses_all,
                        weights=(1, 0, 0, 0))
    # BLEU-4: media geométrica de 1-2-3-4 gramas
    bleu4 = corpus_bleu(references_all, hypotheses_all,
                        weights=(0.25, 0.25, 0.25, 0.25))
    return bleu1, bleu4


print('Calculando BLEU en test (puede tardar ~2-3 min)...')
bleu1, bleu4 = evaluate_bleu(
    model, test_imgs, clean_captions, vocab,
    IMG_DIR, val_transform, device, beam_size=5
)
print(f'\nResultados en test:')
print(f'  BLEU-1: {bleu1:.4f} ({bleu1*100:.2f})')
print(f'  BLEU-4: {bleu4:.4f} ({bleu4*100:.2f})')

## 9. Ejemplos cualitativos

In [ ]:
# ============================================================
# Visualizamos algunos ejemplos del conjunto de test:
# imagen + captions de referencia + generación del modelo
# ============================================================

def show_examples(model, image_names, captions_dict, vocab,
                  img_dir, transform, device, n=6):
    model.eval()
    samples = random.sample(image_names, n)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for ax, img_name in zip(axes, samples):
        # Imagen original (sin normalizar, para visualización)
        img_path = os.path.join(img_dir, img_name)
        image_pil = Image.open(img_path).convert('RGB')
        image_tensor = transform(image_pil)

        greedy_cap = generate_greedy(model, image_tensor, vocab)
        beam_cap   = generate_beam(model, image_tensor, vocab, beam_size=5)

        # Una referencia humana de ejemplo
        ref = clean_captions[img_name][0]

        ax.imshow(image_pil)
        ax.axis('off')
        title = (f'REF:   {ref}\n'
                 f'GREEDY: {greedy_cap}\n'
                 f'BEAM:  {beam_cap}')
        ax.set_title(title, fontsize=7, loc='left', wrap=True)

    plt.suptitle('Ejemplos de generación de captions', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{CHECKPOINT_DIR}/ejemplos.png', dpi=150, bbox_inches='tight')
    plt.show()


show_examples(model, test_imgs, clean_captions, vocab,
              IMG_DIR, val_transform, device, n=6)

In [ ]:
# ============================================================
# EXTRA: prueba con una imagen propia
# Sube cualquier imagen y el modelo generará su descripción
# ============================================================

from google.colab import files

print('Sube una imagen para probar el modelo:')
uploaded = files.upload()

for filename in uploaded.keys():
    image_pil = Image.open(filename).convert('RGB')
    image_tensor = val_transform(image_pil)

    greedy_cap = generate_greedy(model, image_tensor, vocab)
    beam_cap   = generate_beam(model, image_tensor, vocab, beam_size=5)

    plt.figure(figsize=(6, 5))
    plt.imshow(image_pil)
    plt.axis('off')
    plt.title(f'Greedy: {greedy_cap}\nBeam:   {beam_cap}', fontsize=10)
    plt.tight_layout()
    plt.show()

    print(f'Greedy: {greedy_cap}')
    print(f'Beam:   {beam_cap}')